In [32]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [33]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = 'claude-sonnet-4-5'

The Create function is the core of making api requests , requires 3 key parameters - name of the model, max_tokens, the message. may tokemn is a ough limit and claude does not try to hit that limit its an appropriate upper bound



In [34]:
#making my first request
message = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [35]:
#ectractiog the content
message.content[0].text

'Quantum computing harnesses the principles of quantum mechanics—where data can exist in multiple states simultaneously—to perform computations far more efficiently than classical computers for certain types of problems.'

In [51]:
#claude doesnt remember the previous conversations , if u want to have a continuous conversation u need to send the earlier messages back to claude whend
#doing the API call

#Building helper Functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text
   

In [53]:
# putting it all together

# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

print(final_answer)

Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are either 0 or 1.


# System Prompts

In [55]:
# customize how claude responds - 
# basic structure

system_prompt = """ you are a math tutor. do not give the answer directly.guide the student through the solution"""

client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    system=system_prompt
)

Message(id='msg_011CffUThBbqqrhw6sNNEegx', container=None, content=[TextBlock(citations=None, text='Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which can only be either 0 or 1 at any given time.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=81, output_tokens=42, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [59]:
# building a flexible chat function

def chat (messages, system = None):
    params = {
        'model' : 'claude-haiku-4-5',
        'max_tokens' : 1000,
        'messages' : messages,
    }
    if system :
        params['system'] = system

    message = client.messages.create(**params)
    return message.content[0].text    

In [60]:
# without system prompt
answer = chat(messages)

# with the sys prompt

system = """you are a math tutor be patient dont give the answer upright guide the student"""

answer = chat(messages, system=system)

# Temperature
### controls how predictable ot creative the response gonna be

In [62]:
# implementing temperature in code

def chat(messages, system=None, temperature = 1.0):
    params = {
        'model' : 'claude-haiku-4-5',
        'max_tokens' : 1000,
        'messages' : messages,
        'temperature' : temperature
    }

    if system:
        params['system'] = system

    message = client.messages.create(**params)
    return message.content[0].text    

In [ ]:
# testing temperature effects

# low - more predictable
answer1 = chat(messages,temperature=0.0)

# hight - more creative
answer = chat(messages, temperature=1.0)

# At temperature 0.0, you might consistently get responses like "A time-traveling archaeologist must prevent ancient artifacts from being stolen." At temperature 1.0, you'll see much more variety in themes, characters, and plot elements.